# Computing Partial Spectra – A Practical Guide
*Marcel Utz, KIT, 2026*




In order to determine amounts and concentrations, we need the partial spectra of the
compounts of interest. We make the assumption discussed above that the partial spectra
are independent of the amounts. Such partial spectra can be obtained either by 
measurement or by simulation. The former has the advantage that the spectral parameters
do not need to be known in advance. By contrast, experimental spectra are necessarily
measured under specific conditions (solvent, temperature, pH, magnetic field strength) 
that may not correspond to the ones in the measured spectrum. 

A useful compromise is the GISSMO database, which contains the spectral parameters
(chemical shifts and $J$-couplings) for thousands of small molecules. These data have
obviously been derived from experiments, but they allow spin dynamic simulation of the 
partial spectrum at any field strength.


The GISSMO database entries have a unique identifier. To find the identifier corresponding to a specific metabolite, we can use the `GISSMO.search` function:

In [ ]:
import Pkg
Pkg.activate(".")
# using Revise
using NMRlab, Plots
import NMRlab.GISSMO

theme(:dark)
default(size=(600,400),
        tick_direction=:out,
        framestyle=:box,
        margin=(6,:mm),
        legendfontsize=8,
        guidefontsize=10,
        tickfontsize=10,
        fontfamily="Helvetica", 
        fmt=:svg,dpi=100)

metabolite_names = sort(["Glucose", "L-lactate", "L-Alanine", "L-Glutamine",
"L-Glutamic", "L-Asparagine", "L-Valine",
"L-Phenylalanine","HEPES","Arginine","Lysine","L-Leucine",
"L-Isoleucine","L-Proline","L-Serine","L-Threonine","L-Tyrosine","L-Tryptophan",
"L-Cysteine","L-Methionine",
"Taurine","L-Histidine","L-Aspartic","L-Glutathione",
"Creatine","L-Carnitine","Pyruvic", "Acetic", "Glycine", "Choline","Betaine", "Myo-Inositol", "Scyllo-Inositol", "Glycerol", "Ethanol", "Propanol","Butyric"])
gissmo_hits = [GISSMO.search(metabolite) for metabolite in metabolite_names]

Some of these names produced multiple hits. In this case, it makes sense to choose the right one manually. For simplicity, we just use the first hit in every case for the present demo.

In [ ]:
metabolites=[first(k) for k in gissmo_hits]

In order to simulate partial spectra on the same grid, we import the processed experimental
data:

In [ ]:
using JLD2
@load "processedData.jld2" processedData fidData

The GISSMO database contains a spin matrix for each entry. This is a square matrix of dimension $n$, where $n$ is the number of protons
in the molecule that contribute to the NMR spectrum. The diagonal contains the chemical shifts, whereas $J$-couplings between protons
are listed in the off-diagonal elements. Here is HEPES as an example:

In [ ]:
SpM=GISSMO.SpinMatrix(metabolites[11]["id"])

As you can see, there are 16 visible protons in HEPES, represented by a $16\times 16$ spin matrix. If we went about simulating
this, we would be faced with a quantum mechanical system with $2^{16}$ quantum states; operators in Hilbert space
would have to be represented by matrices of size $2^{16} \times 2^{16}$. This is just about doable on a serious computer,
but painful – it would require hours of CPU time.

Fortunately, we do not have to deal with the entire system at once. Closer examination of the spin matrix reveals a block
diagonal structure: not every spin is coupled to all others. We can therefore break the 16-spin system into smaller ones that
are not mutually coupled; therefore they can be simulated separately.


The `NMRlab.GISSMO` library already has a function that reorders the spin matrix into its most block diagonal shape possible. It works by a somewhat different principle, based on finding the connected components of the graph described by using
the non-zero off-diagonal components of the spin matrix as an adjacany matrix.

In [ ]:
blockSpM, blocks = GISSMO.block_diagonal_reorder(SpM)
@show blocks
SpM |> x -> Plots.spy(x,markersize=5,layout=(1,2),size=((800,400)),subplot=1,title="Original")
blockSpM |> x -> Plots.spy!(x,markersize=5,layout=(1,2), subplot=2,title="Block Reordered",color_legend=false)

In [ ]:
using Plots
a=Plots.plot(size=(300*length(blocks),400), layout=(1,length(blocks)), title="Block Diagonal Reordered Hamiltonians", markersize=5,colorbar=false)
for (k,b) in enumerate(blocks)
    n,H=GISSMO.Hamiltonian(SpM[b,b],freq=600.0, ctr=4.78)
    Plots.spy!(H,subplot=k, markersize=4, title="Block $k")
end 
a

We can now use these blocks to compute the frequencies and intensities of the NMR transitions.  The `GISSMO.NMRsignals`function uses a frequency-domain
calculation approach to obtain the quantum transitions and their intensities for each block. From these, we can compute the
spectrum:

In [ ]:
(freqs, ints) = GISSMO.NMRsignals(SpM, freq=600.0, ctr=4.78 )

With these, a time-domain data set can be computed as follows:

In [ ]:
testfid = SpectData(sum(i*exp.(-im*freq*NMRlab.coords(fidData,1)) for (i,freq) in zip(ints,freqs) ),(NMRlab.coords(fidData,1),))

This time-domain data can be processed in the same way as the experimental:

In [ ]:
procSim = Chain(
    ZeroFill([2^16]),
    Apodize([2.5pi]),
    x-> begin x[1]*=0.5; x end,
    FourierTransform([2^16],[1]),
    x->x[10000:end-10000],
)

simspect = procSim(testfid)
plot(NMRlab.coords(simspect,1)./600 .+ 4.78, real.(simspect[:,1]),xaxis=:flip, xlims=(2.7,4.1),)

## Converting a list of metabolites to a decomposition matrix

In practice, we have a list of metabolites that we would like to use to obtain quantitative information (amounts) from
our experimental NMR spectra. In the following, we will generate the partial spectra, assemble a decomposition matrix, and save it in a file
so we can re-use it without having to simulate the partial spectra again.

### List of Spin Matrices
As a first step, we generate a list of metabolites and their associated spin matrices.

In [ ]:
SpMs = Dict(metabolite["name"] => GISSMO.SpinMatrix(metabolite["id"]) for metabolite in metabolites)

In [ ]:
sort(SpMs) 

A few components that we need may not be in the GISSMO database. For example, the chemical shift reference standard (TSP), or co-solvents
such as DMSO. We can manually add their spin matrices, if we know the chemical shifts. Note that TSP contains 9 magnetically equivalent protons. We set its chemical shift to –0.02 ppm. Their
mutual couplings do not influence the spectrum, so we can leave the off-diagonal elements of the spin matrix as zero. The same applies
to DMSO, which has two pairs of 3 magnetically equivalent protons.

In [ ]:
SpMs["TSP"] = Diagonal(zeros(9) .- 0.02)
SpMs["DMSO"]    = Diagonal(zeros(6).+2.71)

In [ ]:
@save "SpMs.jld2" SpMs

### Simulating the partial spectra

Now that we have a list of the necessary spin matrices, we need to use the above quantum dynamics algorithm to simulate the partial spectra.

In [ ]:
partialSpectra = Dict{String,Any}()

for (name, SpM) in sort(SpMs)
    println("Simulating spectrum for ", name)
    freqs,ints = GISSMO.NMRsignals(SpM,freq=600.0,ctr=4.798)
    testfid = SpectData(sum(i*exp.(-im*freq*NMRlab.coords(fidData,1)) for (i,freq) in zip(ints,freqs) ),(NMRlab.coords(fidData,1),))

    partialSpectra[name] = procSim(testfid)

end



In [ ]:
a=plot()  # create an empty plot object to hold all spectra
for (k,(name, spect)) in enumerate(sort(partialSpectra) )
    plot!(a,NMRlab.coords(spect,1)./600 .+ 4.78, real.(spect[:,1]).+(k-1)*2e2,
    xaxis=:flip, ylims=12.0e3.*[-0.1,1.0], label=name,legend=false,size=(800,1000),
    yticks=(0:200:12000,keys(sort(partialSpectra))), xticks=0.5:1.0:7.5,
    xlims=(-0.1,8.5))
end
a

Now that we have laboriously computed these partial spectra, we may want to save them in a file so we can retrieve them later when needed. There are several methods in Julia to save binary and structured objects. We use the `JLD2` package here. Also, it will prove convenient later to have
the spectra in a 2D `SpectData` structure, as well as in a Dictionary where we can access them by name. 

In [ ]:
sort(partialSpectra)

In [ ]:
partialSpects= SpectData(hcat(values(partialSpectra)...),
    (NMRlab.coords(first(values(partialSpectra)),1), 
    collect(keys(partialSpectra))))  

@save "partial_spectra.jld2" partialSpects partialSpectra
partialSpects

### Normalisation

The partial spectra should be normalised in the sense that their integrals must be proportional to the number of spins contributing to each spectrum. Let us verify that this is indeed the case.

In [ ]:
partialSpectsInt = partialSpects |> Integral(1) 

TSPmax = maximum(partialSpectra["TSP"] |> Integral(1) |> real) / 9 

n,m=size(partialSpectsInt)
a=plot()  # create an empty plot object to hold all spectra
for k in 1:m
    plot!(a,NMRlab.coords(partialSpectsInt[:,k],1)./600 .+ 4.78, real.(partialSpectsInt[:,k]) / TSPmax ,
    xaxis=:flip, ylims=20*[-0.1,1.0], label=NMRlab.coords(partialSpectsInt,2)[k],legend=:none,size=(800,600),
    yticks=0:1.0:20.0,linewidth=2,
    xlabel="chemical shift (ppm)", ylabel="integrated intensity (spins)", title="simulated spectra of pure compounds")
end
a